# Optimization Module Test Notebook

This notebook tests the **Optimization** module functionality.

**Test Coverage:**
- OptimizationManager initialization and configuration loading
- Single-objective optimization (STY)
- Multi-objective optimization (weighted sum)
- Desirability-based optimization
- Performance comparison (baseline vs optimized)
- Full export of the optimized simulation (Excel + pickle) to `export/simulations/`

All simulations run in `performance_mode=True` for speed.

## 1. Setup and Imports

In [11]:
import sys
import numpy as np
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from animakinetica.core.simulation import CellCultureSimulation
from animakinetica.core.parameters import SimulationParameters
from animakinetica.models.kinetics import Kinetics
from animakinetica.models.stoichiometry import StoichiometryMode
from animakinetica.models.odesystems import OperatingMode
from animakinetica.models.feed_strategies import FeedStrategy
from animakinetica.models.perfusion_strategies import PerfusionStrategy
from animakinetica.models.gas_controllers import GasController
from animakinetica.analysis.optimization import OptimizationManager, list_opt_configs

print("✓ Imports successful")

✓ Imports successful


## 2. List Available Optimization Configurations

In [12]:
# List all available optimization configurations
available_configs = list_opt_configs()

print(f"✓ Found {len(available_configs)} optimization configurations")
print(f"\nFirst 10 configurations:")
for config in available_configs[:10]:
    print(f"  - {config}")

✓ Found 2 optimization configurations

First 10 configurations:
  - BSF_optimization
  - default_optimization


## 3. Initialize Baseline Simulation

Create a baseline simulation that will be optimized.

In [13]:
# Load base parameters
params = SimulationParameters.load_parameters(["Cell_GFI", "anab_GFI", "cata_GFI", "fed_batch"])# "fed_batch", "perfusion"])

# Initialize baseline simulation with FEDBATCH mode (faster for testing)
baseline_sim = CellCultureSimulation(
    parameters=params,
    kinetics=Kinetics.STOICHIOMETRIC,
    stoichiometry_mode=StoichiometryMode.CONSTANT,
    operating_mode=OperatingMode.BATCH_WITH_STARTUP_FEDBATCH,
    feed_strategy=FeedStrategy.EXPONENTIAL,
    #perfusion_strategy=PerfusionStrategy.CONSTANT,
    gas_controller=GasController.CASCADE_N_FG_XO2,
    performance_mode=False  # Use performance mode for faster optimization
)

# Run baseline simulation
baseline_result = baseline_sim.run_simulation()

print(f"✓ Baseline simulation completed")
print(f"\nBaseline Performance Metrics:")
print(f"  STY: {baseline_sim.data.summary_stats['STY_kg_m3_day']:.4f} kg/m³/day")
print(f"  Total DCM: {baseline_sim.data.projected_totals_stats['total_DCM_out_kg']:.4f} kg")
print(f"  Medium Conversion Efficiency: {baseline_sim.data.summary_stats['medium_conversion_efficiency_m3_kg']:.3f} m3/kg DCM")
print(f"  Protein Conversion Efficiency: {baseline_sim.data.summary_stats['protein_conversion_efficiency_kg_kg']:.3f} kg/kg")
print(f"  Feed Conversion Efficiency: {baseline_sim.data.summary_stats['feed_conversion_efficiency_kg_kg']:.3f} kg/kg")
print(f"  Total Simulation Time: {baseline_sim.data.projected_totals_stats['total_simulation_time_h']:.2f} h")
print(f"  Number of Batches: {baseline_sim.data.projected_totals_stats['number_of_batches']}")

Loading parameters from 5 file(s) sequentially...
  [1/5] Base: c:\Users\btumu\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\base_parameters.yaml
  [2/5] Updating with: c:\Users\btumu\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\cell_properties\Cell_GFI.yaml
  [3/5] Updating with: c:\Users\btumu\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\stoichiometry\anab_GFI.yaml
  [4/5] Updating with: c:\Users\btumu\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\stoichiometry\cata_GFI.yaml
  [5/5] Updating with: c:\Users\btumu\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\operating_modes\fed_batch.yaml
Successfully loaded parameters from 5 file(s).
run_simulation execution time: 728.575 ms
✓ Baseline simulation 

## 4. Initialize OptimizationManager with Default Config

In [14]:
# Create optimization manager with default configuration
opt_manager = OptimizationManager(
    simulation=baseline_sim,
    opt_config_file="BSF_optimization" # None  # Uses default config
)

print(f"✓ OptimizationManager initialized with default config")
print(f"  Configuration file: {opt_manager.opt_config_file}")
print(f"  Number of parameters to optimize: {len(opt_manager.opt_config)}")

# Display optimization parameters
print(f"\nOptimization Parameters:")
for i, (param_name, config) in enumerate(opt_manager.opt_config.items(), 1):
    bounds = config['bounds']
    label = config['label']
    unit = config['unit']
    print(f"  {i}. {param_name}: [{bounds[0]:.4g}, {bounds[1]:.4g}] {unit} ({label})")

Searching specified optimization config: BSF_optimization
Loading optimization config from: c:\Users\btumu\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\optimization_config\BSF_optimization.yaml
Parsing...
Parsed and loaded configuration containing 1 optimization parameters.
✓ OptimizationManager initialized with default config
  Configuration file: BSF_optimization
  Number of parameters to optimize: 1

Optimization Parameters:
  1. initial_conditions.C_Glc_init: [1, 100]  (C_Glc_init)


## 6. Single-Objective Optimization Test

Optimize for a single objective (STY = Space-Time Yield).

In [15]:
# Run single-objective optimization (fast: performance-mode manager + few iterations)
print("Running single-objective optimization for STY...")

opt_manager.optimize_parameters(
    objectives=["STY_kg_m3_day"],
    workers=-1,
    verbose=False
)

print(f"✓ Single-objective optimization completed ({opt_manager.optimization_result.nit} iterations)")
print(f"\nOptimal Parameters Found:")
for param_name, value in opt_manager.optimal_parameters.items():
    config = opt_manager.opt_config[param_name]
    print(f"  {param_name}: {value:.4g} {config['unit']} ({config['label']})")

Running single-objective optimization for STY...
run_simulation execution time: 498.063 ms


KeyboardInterrupt: 

## 7. Performance Comparison: Baseline vs Optimized

In [ ]:
# Compare baseline and optimized performance
comparison_df = opt_manager.compare_performance(verbose=True)


=== PERFORMANCE COMPARISON ===
                          Metric                                                                                  Description    Baseline                 Unit   Optimized  Improvement
      Feed Conversion Efficiency         Glucose, glutamine, and amino-acid input per harvested dry biomass; lower is better.    4.695764 kg Glc+Gln+AA/kg DCM    6.191727     1.495963
               Harvested Biomass   Productive dry biomass harvested over the full simulation, excluding non-harvested losses.  209.646871                   kg  213.796953     4.150082
           Harvested Wet Biomass Productive wet cell mass harvested over the full simulation, excluding non-harvested losses. 1397.645809               kg WCM 1425.313020    27.667211
    Medium Conversion Efficiency                       Total process medium input per harvested dry biomass; lower is better.    0.722117            m3/kg DCM    0.637850    -0.084267
   Protein Conversion Efficiency              Gl

## 8. Multi-Objective Optimization Test

Optimize for multiple objectives with weighted sum approach.

In [16]:
# Reuse the existing baseline simulation instead of building a new one
opt_manager_multi = OptimizationManager(
    simulation=baseline_sim,
    opt_config_file=None
)

print("Running multi-objective optimization...")
print("Objectives: STY (70%), Feed Conversion Efficiency (30%)\n")

opt_manager_multi.optimize_parameters(
    objectives=["STY_kg_m3_day", "feed_conversion_efficiency_kg_kg"],
    weights=[0.7, 0.3],
    workers=-1,
    verbose=False
)

print(f"✓ Multi-objective optimization completed")

No optimization config specified. Using default...
Loading optimization config from: c:\Users\btumu\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\optimization_config\default_optimization.yaml
Parsing...
Parsed and loaded configuration containing 2 optimization parameters.
Running multi-objective optimization...
Objectives: STY (70%), Feed Conversion Efficiency (30%)

run_simulation execution time: 543.383 ms


KeyboardInterrupt: 

## 9. Multi-Objective Performance Comparison

In [ ]:
# Compare multi-objective results
comparison_multi_df = opt_manager_multi.compare_performance(verbose=True)

# Analyze trade-offs
print("\n=== MULTI-OBJECTIVE TRADE-OFF ANALYSIS ===")
for metric in ['STY_kg_m3_day', 'feed_conversion_efficiency_kg_kg', 'medium_conversion_efficiency_m3_kg']:
    if metric in comparison_multi_df.index:
        baseline = comparison_multi_df.loc[metric, 'Baseline']
        optimized = comparison_multi_df.loc[metric, 'Optimized']
        improvement_pct = (optimized - baseline) / baseline * 100
        print(f"{metric}:")
        print(f"  Change: {improvement_pct:+.2f}%")


=== PERFORMANCE COMPARISON ===
                          Metric                                                                                  Description    Baseline                 Unit   Optimized  Improvement
      Feed Conversion Efficiency         Glucose, glutamine, and amino-acid input per harvested dry biomass; lower is better.    4.852275 kg Glc+Gln+AA/kg DCM    3.781302    -1.070973
               Harvested Biomass   Productive dry biomass harvested over the full simulation, excluding non-harvested losses.  223.659691                   kg  223.347687    -0.312004
           Harvested Wet Biomass Productive wet cell mass harvested over the full simulation, excluding non-harvested losses. 1491.064606               kg WCM 1488.984579    -2.080027
    Medium Conversion Efficiency                       Total process medium input per harvested dry biomass; lower is better.    0.743324            m3/kg DCM    0.677117    -0.066207
   Protein Conversion Efficiency              Gl

# 10. Test Optimization with Engineering Desirability Scoring


In [17]:
# Test Optimization with Engineering Desirability Scoring
print("Running optimization with Engineering Desirability scoring...")

# Baseline metrics as reference
base_sty = baseline_sim.data.summary_stats['STY_kg_m3_day']
base_med_eff = baseline_sim.data.summary_stats['medium_conversion_efficiency_m3_kg']

# Desirability config:
# - STY (maximize): minimum acceptable is 50% of baseline, target is 150% of baseline
# - medium_conversion_efficiency_m3_kg (minimize): minimum unacceptable is 150% of baseline, target is 70% of baseline
desirability_config = {
    'STY_kg_m3_day': {
        'minimum': base_sty * 0.5,
        'target': base_sty * 1.5,
        'safety_cap': 1.5
    },
    'medium_conversion_efficiency_m3_kg': {
        'minimum': base_med_eff * 1.5,
        'target': base_med_eff * 0.7,
        'safety_cap': 1.5
    }
}

opt_manager_desirability = OptimizationManager(
    simulation=baseline_sim,
    opt_config_file="BSF_optimization"
)

opt_manager_desirability.optimize_parameters(
    objectives=['STY_kg_m3_day', 'medium_conversion_efficiency_m3_kg'],
    weights=[0.6, 0.4],
    desirability_config=desirability_config,
    scoring_profile="test_desirability_profile",
    workers=-1,
    verbose=False
)

print("✓ Desirability optimization completed")

Running optimization with Engineering Desirability scoring...
Searching specified optimization config: BSF_optimization
Loading optimization config from: c:\Users\btumu\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\optimization_config\BSF_optimization.yaml
Parsing...
Parsed and loaded configuration containing 1 optimization parameters.
run_simulation execution time: 581.119 ms
run_simulation execution time: 454.150 ms
optimize_parameters execution time: 88652.105 ms
✓ Desirability optimization completed


In [18]:
comparison_desirability_df = opt_manager_desirability.compare_performance(verbose=True)


=== PERFORMANCE COMPARISON ===
                          Metric                                                                                  Description    Baseline                 Unit   Optimized  Improvement
      Feed Conversion Efficiency         Glucose, glutamine, and amino-acid input per harvested dry biomass; lower is better.    4.695764 kg Glc+Gln+AA/kg DCM    6.069626     1.373861
               Harvested Biomass   Productive dry biomass harvested over the full simulation, excluding non-harvested losses.  209.646869                   kg  213.022106     3.375237
           Harvested Wet Biomass Productive wet cell mass harvested over the full simulation, excluding non-harvested losses. 1397.645793               kg WCM 1420.147374    22.501580
    Medium Conversion Efficiency                       Total process medium input per harvested dry biomass; lower is better.    0.722117            m3/kg DCM    0.640171    -0.081947
   Protein Conversion Efficiency              Gl

## 11. Test Optimization Result Storage and Access

In [19]:
# Test that optimization results are properly stored
print("Optimization Result Attributes:")
print(f"  Success: {opt_manager.optimization_result.success}")
print(f"  Message: {opt_manager.optimization_result.message}")
print(f"  Number of iterations: {opt_manager.optimization_result.nit}")
print(f"  Number of function evaluations: {opt_manager.optimization_result.nfev}")
print(f"  Optimal objective value: {-opt_manager.optimization_result.fun:.4f}")  # Negative because minimization

# Verify optimized simulation exists and has data
print(f"\n✓ Optimized simulation verification:")
print(f"  Data object exists: {opt_manager.optimized_simulation.data is not None}")
print(f"  Number of batches: {len(opt_manager.optimized_simulation.data.batches)}")
print(f"  Summary stats available: {len(opt_manager.optimized_simulation.data.summary_stats) > 0}")

Optimization Result Attributes:


AttributeError: 'NoneType' object has no attribute 'success'

## 11. Test Parameter Bounds Processing

In [20]:
# Test bounds processing for different bound types
print("Testing Parameter Bounds Processing:")

# Check that all bounds are valid tuples
bounds_valid = True
for param_name, config in opt_manager.opt_config.items():
    bounds = config['bounds']
    if not isinstance(bounds, tuple) or len(bounds) != 2:
        print(f"  ✗ Invalid bounds for {param_name}: {bounds}")
        bounds_valid = False
    elif bounds[0] >= bounds[1]:
        print(f"  ✗ Invalid bounds range for {param_name}: {bounds}")
        bounds_valid = False

if bounds_valid:
    print(f"✓ All {len(opt_manager.opt_config)} parameter bounds are valid")
else:
    print("✗ Some parameter bounds are invalid")

# Display bounds statistics
all_bounds = [config['bounds'] for config in opt_manager.opt_config.values()]
lower_bounds = [b[0] for b in all_bounds]
upper_bounds = [b[1] for b in all_bounds]

print(f"\nBounds Statistics:")
print(f"  Lower bounds range: [{min(lower_bounds):.2e}, {max(lower_bounds):.2e}]")
print(f"  Upper bounds range: [{min(upper_bounds):.2e}, {max(upper_bounds):.2e}]")

Testing Parameter Bounds Processing:
✓ All 1 parameter bounds are valid

Bounds Statistics:
  Lower bounds range: [1.00e+00, 1.00e+00]
  Upper bounds range: [1.00e+02, 1.00e+02]


## 14. Test Error Handling

In [21]:
# Test error handling for invalid inputs (validation happens before any simulation runs)
print("Testing error handling:")

# Test 1: Mismatched objectives and weights
try:
    opt_manager.optimize_parameters(
        objectives=["STY_kg_m3_day", "medium_efficiency_kg_m3"],
        weights=[1.0],  # Only one weight for two objectives
        workers=1,
        verbose=False
    )
    print("  ✗ Test 1 failed: Should raise ValueError for mismatched objectives/weights")
except ValueError as e:
    print(f"  ✓ Test 1 passed: ValueError raised as expected")
    print(f"    Message: {str(e)[:80]}...")

# Test 2: Empty objectives list
try:
    opt_manager.optimize_parameters(
        objectives=[],
        weights=[],
        workers=1,
        verbose=False
    )
    print("  ✗ Test 2 failed: Should raise ValueError for empty objectives")
except ValueError as e:
    print(f"  ✓ Test 2 passed: ValueError raised as expected")
    print(f"    Message: {str(e)[:80]}...")

# Test 3: Optimization manager without parameters
try:
    sim_no_params = CellCultureSimulation(
        parameters=None,
        kinetics=Kinetics.STOICHIOMETRIC,
        operating_mode=OperatingMode.BATCH
    )
    opt_manager_invalid = OptimizationManager(sim_no_params)
    print("  ✗ Test 3 failed: Should raise ValueError for simulation without parameters")
except (ValueError, AttributeError) as e:
    print(f"  ✓ Test 3 passed: Exception raised as expected")
    print(f"    Message: {str(e)[:80]}...")

print("\n✓ Error handling tests completed")

Testing error handling:
  ✓ Test 1 passed: ValueError raised as expected
    Message: Number of objectives (2) must match number of weights (1)...
  ✓ Test 2 passed: ValueError raised as expected
    Message: At least one objective must be specified...
  ✓ Test 3 passed: Exception raised as expected
    Message: 'NoneType' object has no attribute 'create_mode_adjusted_copy'...

✓ Error handling tests completed


## 11. Export Optimized Simulation

Export the optimized simulation with a complete Excel export (including pickle and parameters YAML) to `export/simulations/`.

In [24]:
from pathlib import Path

# Complete export of the optimized simulation:
# - Excel workbook (metadata, summary stats, batch data, parameters)
# - Pickle file for fast reload later
# - Parameters YAML
opt_sim = opt_manager_desirability.optimized_simulation

# Package default export root: <repo>/export/simulations/
export_dir = Path.cwd().parent / "export" / "simulations"

opt_sim.save(
    filename="optimization_test_optimized",
    file_format="excel",
    add_timestamp=True,
    include_metadata=True,
    include_summary_stats=True,
    include_cultivation_summary=True,
    include_batch_data=True,
    include_parameters=True,
    include_pickle=True,
    overwrite=True,
    export_dir=export_dir
)

print(f"✓ Optimized simulation exported to: {export_dir}")
print(f"  STY: {opt_sim.data.summary_stats['STY_kg_m3_day']:.4f} kg/m³/day")
print(f"  Batches: {len(opt_sim.data.batches)}")

Simulation data exported to: c:\Users\btumu\OneDrive - Delft University of Technology\2. Code\animakinetica\export\simulations\optimization_test_optimized_20260926_2123\opt_20260926_2123.xlsx
Parameters saved to: c:\Users\btumu\OneDrive - Delft University of Technology\2. Code\animakinetica\export\simulations\optimization_test_optimized_20260926_2123\opt_20260926_2123.parameters.yaml
CellCultureSimulation pickle saved to: c:\Users\btumu\OneDrive - Delft University of Technology\2. Code\animakinetica\export\simulations\optimization_test_optimized_20260926_2123\opt_20260926_2123.pkl
✓ Optimized simulation exported to: c:\Users\btumu\OneDrive - Delft University of Technology\2. Code\animakinetica\export\simulations
  STY: 0.3328 kg/m³/day
  Batches: 9
